In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import re
import numpy as np
import pandas as pd

from kalshi_core import KalshiClient, pnl_summary, portfolio_kelly, kalshi_order_price
from src.data import get_username_mappings, load_and_prepare, load_model_predictions
from src.kalshi_tt import get_tt_asks
from src.simulation import build_player_pool
from src.portfolio import (
    build_portfolio, run_portfolio_mc,
    run_payoff_matrix,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 300)
pd.set_option('display.max_colwidth', 300)

USERNAME_TO_PLAYER, PLAYER_TO_USERNAME = get_username_mappings()

_results_raw = load_model_predictions().set_index('username')
for _n in [1, 3, 5, 8, 10]:
    _results_raw[f'P_top{_n}'] = _results_raw['p_participate'] * _results_raw[f'P_top{_n}_given_play']
results = _results_raw

In [ ]:
def display_profits(df_tt, df_results, verbose=False):
    costs = 0
    revenue = 0
    for idx, row in df_tt.iterrows():
        costs += row['cost']
        pos = 1.0 if row['position']=='Yes' else 0.0
        username = PLAYER_TO_USERNAME[row['marketTitle']]
        n = row['n']
        if not username in df_results.index:
            if verbose:
                print(f"{row['marketTitle']} in top {n}: -${row['cost']}")
            continue
        result = df_results.loc[username, f'P_top{n}'].item()
        if result == pos:
            if verbose:
                print(f"{row['marketTitle']} in top {n}: {'YES' if result > 0 else 'NO'}. +${row['volume'] - row['cost']}")
            revenue += row['volume']
        else:
            if verbose:
                print(f"{row['marketTitle']} in top {n}: {'YES' if result > 0 else 'NO'}. -${row['cost']}")
    return revenue - costs

def avg_profits(df, df_results, fees=0, verbose=False):
    revenues = []
    costs = []
    values = []
    values_per_share = []
    for idx, row in df.iterrows():
        username = PLAYER_TO_USERNAME[row['marketTitle']]
        n = row['n']
        costs.append(round(row['volume']*row['orderPrice'],2))
        if not username in df_results.index:
            print(f"{row['marketTitle']} in top {n}\t\t{row['position'].upper()}:\t0.0 EV per share.")
            revenues.append(0)
            continue
        result = df_results.loc[username, f'P_top{n}'].item()
        value = result if row['position']=='Yes' else (1-result)
        revenues.append(row['volume']*value)
        values.append(f'${value*row["volume"]:.2f}')
        values_per_share.append(value)
        if verbose:
            print(f"{row['marketTitle']} in top {n}\t\t{row['position'].upper()}:\t{value:.2f} EV per share.\t ${row['volume']*value-row['cost']:.2f} EV\t {(row['volume']*value/row['cost']-1)*100:.2f}% ROI")
    expected_return_col = [round(r-c,2) for r,c in zip(revenues, costs)]
    roi_col = [round(r/c,2) for r,c in zip(revenues, costs)]
    revenue = sum(revenues)
    cost = sum(costs)+fees
    net_profit = revenue-cost
    roi = (revenue/cost-1)*100
    print(f"Total Exposure:{cost}. Expected Revenue: {revenue}. Net profits avg: {net_profit:.2f}. ROI {roi:.2f}%")
    return expected_return_col, values, values_per_share, roi_col, costs

n_title_dict = {
    'Titled Tuesday Top 8: Jul 21': 8,
    'Titled Tuesday Top 3: Jul 21': 3,
    'Titled Tuesday Winner: Jul 21': 1
    }

kalshi_orders = pd.read_csv("../data/Kalshi-Advanced-Portfolio.csv")
df_tt = kalshi_orders[kalshi_orders['eventTitle'].str.contains('Tuesday')]
df_tt[['position','volume']] = df_tt['position'].str.split(' x ',expand=True)
df_tt['volume'] = df_tt['volume'].astype(float)
df_tt['n'] = df_tt['eventTitle'].apply(lambda x: n_title_dict[x])
df_tt['orderPrice'] = kalshi_order_price(df_tt['averagePrice'].str.extract('(\d+)').astype(float)/100.0)
df_tt['expectedReturn'], df_tt['expectedValue'], df_tt['evPerShare'], df_tt['ROI'], df_tt['cost']=avg_profits(df_tt, results)
df_tt['eventTitle'] = df_tt['eventTitle'].apply(lambda event: f'Top {n_title_dict[event]}')
display(df_tt[['eventTitle','marketTitle','position','ROI','orderPrice','evPerShare','volume','cost','expectedValue','expectedReturn']].sort_values('ROI',ascending=False))

df, p_participate, app_counts = load_and_prepare(
    scheduling_conflict=['Matthias Bluebaum','Levon Aronian','Yagiz Kaan Erdogmus','Jose Martinez','Le Quang Liem', 'Daniel Naroditsky'],
)
players, p_play, hist_pcts, hist_wts = build_player_pool(df, p_participate, app_counts)
portfolio = build_portfolio(df_tt, players, PLAYER_TO_USERNAME)
pnl = run_portfolio_mc(players, p_play, hist_pcts, hist_wts, portfolio, n_sims=20_000)
pnl_summary(pnl)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sns.histplot(x=pnl, kde=True, ax=ax, color='blue', bins=500)
ax.set_title('Histogram & KDE Plot of EV Distribution from Kalshi TT Portfolio')
plt.show()
plt.close(fig)

In [ ]:
TOURN_DATE = "2026-07-21"
client = KalshiClient()
all_asks = get_tt_asks(client, TOURN_DATE)
print(f"Found {len(all_asks)} orderbook levels for {TOURN_DATE}.")

In [ ]:
_results_raw = load_model_predictions().set_index('username')
for _n in [1, 3, 5, 8, 10]:
    _results_raw[f'P_top{_n}'] = _results_raw['p_participate'] * _results_raw[f'P_top{_n}_given_play']
    _results_raw[f'ip_adv_top{_n}'] = _results_raw[f'P_top{_n}_given_play'] / _results_raw[f'P_top{_n}']
results = _results_raw
dfk = pd.DataFrame(all_asks)
dfk['n'] = dfk['Market Title'].apply(lambda s: re.search(r'\d+',s).group())

dfk['username'] = dfk['Player Name'].apply(lambda n: PLAYER_TO_USERNAME[n])
evs = []
order_prices=[]
for _, row in dfk.iterrows():
    n = 1 if row['n'] == '21' else int(row['n'])
    p_yes = (results.loc[row['username'], f'P_top{n}'].item())
    evs.append(p_yes if row['Side']=='YES' else (1-p_yes))

dfk['EV'] = evs
dfk['Order Price'] = (dfk['Ask Price ($)'] + 0.07 * dfk['Ask Price ($)'] * (1.0 - dfk['Ask Price ($)']))
dfk['Total Cost'] = dfk['Quantity Available']*dfk['Order Price']
dfk['Expected Return'] = ((dfk['EV']-dfk['Order Price'])*dfk['Quantity Available'].astype(int))
dfk['ROI'] = round(dfk['EV']/dfk['Order Price'].astype(float),3)
dfk = dfk[dfk['ROI']>1.]
print(dfk['Expected Return'].sum())
dfk[['Market Title','ROI','Side','Order Price','EV','Quantity Available','Total Cost','Expected Return']][(dfk['ROI']>1.)].sort_values('ROI',ascending=False)

261.43298176962185


,Market Title,ROI,Side,Order Price,EV,Quantity Available,Total Cost,Expected Return
1296,"Will Hikaru Nakamura finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",1.245,NO,0.597052,0.743074,120.39,71.879090,17.522598
1295,"Will Hikaru Nakamura finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",1.224,NO,0.606933,0.743074,200.00,121.386600,27.228130
939,"Will Sina Movahed finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",1.218,YES,0.116853,0.142284,126.00,14.723478,3.204313
2399,"Will Denis Lazavik finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",1.216,YES,0.304413,0.370025,1.00,0.304413,0.065612
2207,"Will Jan-Krzysztof Duda finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",1.202,YES,0.263125,0.316375,25.00,6.578125,1.331249
1515,"Will Aleksei Sarana finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",1.196,YES,0.148428,0.177498,314.00,46.606392,9.127846
1910,"Will Parham Maghsoodloo finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",1.196,YES,0.221613,0.265091,1.00,0.221613,0.043478
1398,"Will Denis Lazavik finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",1.193,YES,0.158925,0.189529,291.00,46.247175,8.905875
2549,"Will Aleksei Sarana finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",1.177,YES,0.335232,0.394445,31.00,10.392192,1.835591
2398,"Will Denis Lazavik finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",1.176,YES,0.314700,0.370025,31.00,9.755700,1.715074


In [ ]:
# ── Simple (single-bet) Kelly sizing ─────────────────────────────────────────
# f* = (p - c) / (1 - c)  where p = model probability, c = cost per share
# Applied independently per position then normalised if total > 1.
# Use KELLY_MULT < 1 (half-Kelly = 0.5 recommended) to account for model error.

BANKROLL   = 1710   # <-- set to your actual bankroll in $
KELLY_MULT = 0.5    # half-Kelly

df_pos = dfk[dfk['ROI'] > 1.0].copy()

# Raw single-bet Kelly fraction: optimal bankroll fraction if this were the only bet
df_pos['kelly_f'] = (df_pos['EV'] - df_pos['Order Price']) / (1.0 - df_pos['Order Price'])

# If the sum of independent Kelly fractions exceeds 1, the positions collectively
# need more than the full bankroll.  Normalise proportionally as a conservative
# approximation for correlated simultaneous bets.
total_f = df_pos['kelly_f'].sum()
if total_f > 1.0:
    print(f"Sum of single-bet Kelly fractions = {total_f:.2f} — normalising to 1.0")
    df_pos['kelly_f'] /= total_f

# alloc_frac: final fraction of bankroll deployed to each position
# = normalised kelly_f * KELLY_MULT
df_pos['alloc_frac']   = df_pos['kelly_f'] * KELLY_MULT
df_pos['kelly_shares'] = (df_pos['alloc_frac'] * BANKROLL / df_pos['Order Price']).round().clip(lower=0).astype(int)
df_pos['kelly_cost']   = df_pos['kelly_shares'] * df_pos['Order Price']
df_pos['kelly_ev']     = df_pos['kelly_shares'] * df_pos['EV']
df_pos['kelly_edge']   = df_pos['kelly_ev'] - df_pos['kelly_cost']

df_kelly = df_pos[df_pos['kelly_shares'] > 0].copy()

total_cost = df_kelly['kelly_cost'].sum()
total_ev   = df_kelly['kelly_ev'].sum()
print(f"Bankroll ${BANKROLL:,.0f}  |  Exposure ${total_cost:.0f} ({total_cost/BANKROLL*100:.1f}%)  |  "
      f"Expected edge ${total_ev - total_cost:.2f}  |  Portfolio ROI {(total_ev/total_cost - 1)*100:.1f}%\n")

df_kelly[['Market Title', 'Side', 'ROI', 'Order Price', 'EV', 'kelly_f', 'alloc_frac', 'kelly_shares', 'kelly_cost', 'kelly_edge']]\
    .sort_values('kelly_f', ascending=False)\
    .style.format({'Order Price': '{:.3f}', 'EV': '{:.3f}', 'kelly_f': '{:.4f}', 'alloc_frac': '{:.4f}',
                   'kelly_cost': '${:.2f}', 'kelly_edge': '${:.2f}'})

Sum of single-bet Kelly fractions = 20.51 — normalising to 1.0
Bankroll $1,710  |  Exposure $855 (50.0%)  |  Expected edge $42.62  |  Portfolio ROI 5.0%



,Market Title,Side,ROI,Order Price,EV,kelly_f,alloc_frac,kelly_shares,kelly_cost,kelly_edge
2078,"Will Levon Aronian finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.039000,0.963,1.000,0.0488,0.0244,43,$41.40,$1.60
2077,"Will Levon Aronian finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.029000,0.972,1.000,0.0488,0.0244,43,$41.80,$1.20
857,"Will Shamsiddin Vokhidov finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.034000,0.963,0.996,0.0433,0.0216,38,$36.58,$1.26
856,"Will Shamsiddin Vokhidov finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.024000,0.972,0.996,0.0415,0.0207,36,$34.99,$0.86
1647,"Will Vincent Keymer finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.049000,0.944,0.990,0.0405,0.0202,37,$34.93,$1.72
1669,"Will Vladislav Artemiev finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.101000,0.887,0.977,0.0388,0.0194,37,$32.83,$3.32
1646,"Will Vincent Keymer finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.039000,0.953,0.990,0.0388,0.0194,35,$33.37,$1.30
1723,"Will Sergey Sklokin finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.015000,0.981,0.996,0.0385,0.0192,34,$33.37,$0.50
1668,"Will Vladislav Artemiev finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.090000,0.897,0.977,0.0379,0.0190,36,$32.29,$2.89
941,"Will Rasmus Svane finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.021000,0.972,0.993,0.0359,0.0179,32,$31.11,$0.66


In [ ]:
df, p_participate, app_counts = load_and_prepare(
    scheduling_conflict=[
    'Matthias Bluebaum',
    'Levon Aronian',
    'Yagiz Kaan Erdogmus',
    'Jose Martinez Alcantara',
    'Le Quang Liem',
    'Hans Niemann',
    'Arjun Erigaisi',
    'Pranesh Munirethinam',
    'Dmitry Andreikin',
    'Nihal Sarin',
    'Alireza Firouzja',
    'Nodirbek Abdusattorov',
],
)
players, p_play, hist_pcts, hist_wts = build_player_pool(df, p_participate, app_counts)

In [ ]:
df_cands = dfk[dfk['ROI'] > 1.0].copy()
df_cands['n_int'] = df_cands['n'].apply(lambda x: 1 if str(x) == '21' else int(x))

df_cands_fmt = (
    df_cands
    .drop(columns=['n'])
    .rename(columns={'Player Name': 'marketTitle', 'Side': 'position', 'n_int': 'n'})
    .copy()
)
df_cands_fmt['position'] = df_cands_fmt['position'].str.title()
df_cands_fmt['volume']   = 1.0
df_cands_fmt['cost']     = df_cands_fmt['Order Price']

portfolio_cands = build_portfolio(df_cands_fmt, players, PLAYER_TO_USERNAME)

print("Running payoff matrix simulation (50k sims)...")
payoff_matrix, cost_per_share = run_payoff_matrix(
    players, p_play, hist_pcts, hist_wts,
    portfolio_cands, n_sims=50_000,
)

print("\nOptimising Kelly allocation...")
opt_shares, opt_result = portfolio_kelly(
    payoff_matrix, cost_per_share,
    bankroll=BANKROLL, kelly_fraction=KELLY_MULT,
)

df_cands['opt_shares'] = opt_shares
df_cands['opt_cost']   = opt_shares * df_cands['Order Price']
df_cands['opt_ev']     = opt_shares * df_cands['EV']
df_cands['opt_edge']   = df_cands['opt_ev'] - df_cands['opt_cost']

df_opt = df_cands[df_cands['opt_shares'] > 0].copy()
total_cost_opt = df_opt['opt_cost'].sum()
total_ev_opt   = df_opt['opt_ev'].sum()
print(f"\nPortfolio Kelly  |  Exposure ${total_cost_opt:.0f} ({total_cost_opt/BANKROLL*100:.1f}%)  |  "
      f"Expected edge ${total_ev_opt - total_cost_opt:.2f}  |  ROI {(total_ev_opt/total_cost_opt - 1)*100:.1f}%\n")

df_opt[['Market Title', 'Side', 'ROI', 'Order Price', 'EV', 'opt_shares', 'opt_cost', 'opt_edge']]\
    .sort_values('opt_shares', ascending=False)\
    .style.format({'Order Price': '{:.3f}', 'EV': '{:.3f}',
                   'opt_cost': '${:.2f}', 'opt_edge': '${:.2f}'})

Running payoff matrix simulation (50k sims)...
   10,000 / 50,000  (5.2s)


KeyboardInterrupt: 

In [ ]:
# ── A: Recommended additional buys to reach Kelly-optimal sizing ──────────────
# Compares the portfolio Kelly target (opt_shares) for each position against
# shares already held in df_tt, then recommends the difference.
# Positions not yet held at all are treated as 0 shares held.

# Sum existing shares per (player_name, n, side) across all df_tt rows
held = (
    df_tt
    .assign(side_up=df_tt['position'].str.upper())
    .groupby(['marketTitle', 'n', 'side_up'])['volume']
    .sum()
)  # MultiIndex Series: (marketTitle, n, side_up) -> total shares held

def shares_held(player_name, n_int, side):
    try:
        return held.loc[(player_name, n_int, side)]
    except KeyError:
        return 0.0

df_buys = df_opt.copy()
df_buys['held']              = df_buys.apply(lambda r: shares_held(r['Player Name'], r['n_int'], r['Side']), axis=1)
df_buys['additional_shares'] = (df_buys['opt_shares'] - df_buys['held']).clip(lower=0).astype(int)
df_buys['additional_cost']   = df_buys['additional_shares'] * df_buys['Order Price']
df_buys['additional_ev']     = df_buys['additional_shares'] * df_buys['EV']
df_buys['additional_edge']   = df_buys['additional_ev'] - df_buys['additional_cost']

df_new_buys = df_buys[df_buys['additional_shares'] > 0].copy()

total_new_cost = df_new_buys['additional_cost'].sum()
total_new_ev   = df_new_buys['additional_ev'].sum()
print(f"Recommended buys  |  Additional exposure ${total_new_cost:.0f}  |  "
      f"Expected edge ${total_new_ev - total_new_cost:.2f}  |  "
      f"ROI {(total_new_ev / total_new_cost - 1)*100:.1f}%\n")

df_new_buys[['Market Title', 'Side', 'ROI', 'Order Price', 'EV', 'held', 'opt_shares', 'additional_shares', 'additional_cost', 'additional_edge']]\
    .sort_values('additional_edge', ascending=False)\
    .style.format({'Order Price': '{:.3f}', 'EV': '{:.3f}',
                   'additional_cost': '${:.2f}', 'additional_edge': '${:.2f}'})

Recommended buys  |  Additional exposure $267  |  Expected edge $33.87  |  ROI 12.7%



,Market Title,Side,ROI,Order Price,EV,held,opt_shares,additional_shares,additional_cost,additional_edge
1075,"Will Jose Martinez finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.103000,0.906,1.000,200.880000,375,174,$157.70,$16.30
724,"Will Denis Lazavik finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",YES,1.277000,0.148,0.190,171.000000,321,150,$22.26,$6.17
845,"Will Vladislav Artemiev finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,1.099000,0.887,0.976,66.000000,131,65,$57.68,$5.74
1205,"Will Dmitry Andreikin finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",YES,1.249000,0.190,0.238,372.000000,473,101,$19.22,$4.80
553,"Will Parham Maghsoodloo finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",YES,1.088000,0.106,0.116,13.000000,70,57,$6.06,$0.53
260,"Will Jan-Krzysztof Duda win the Titled Tuesday weekly chess competition, originally scheduled for Jul 21, 2026 at 11:00am EDT?",YES,1.087000,0.064,0.070,0.000000,58,58,$3.71,$0.32


In [ ]:
# ── B: Maker bid orders for recommended buys ─────────────────────────────────
# Places limit orders 1 cent below the current ask, qualifying for the lower
# maker fee (1.75% vs 7%).  Drops any position where the lower bid price
# makes the trade negative EV after the maker fee is applied.

df_maker = df_new_buys.copy()
df_maker['maker_ask']         = (df_maker['Ask Price ($)'] - 0.01).clip(lower=0.01).round(2)
df_maker['maker_order_price'] = kalshi_order_price(df_maker['maker_ask'], make=True)
df_maker['maker_roi']         = df_maker['EV'] / df_maker['maker_order_price']
df_maker['maker_edge_share']  = df_maker['EV'] - df_maker['maker_order_price']
df_maker['maker_total_cost']  = df_maker['additional_shares'] * df_maker['maker_order_price']
df_maker['maker_total_edge']  = df_maker['additional_shares'] * df_maker['maker_edge_share']

# Taker vs maker cost savings per position
df_maker['fee_saving']        = df_maker['additional_shares'] * (df_maker['Order Price'] - df_maker['maker_order_price'])

df_maker = df_maker[df_maker['maker_roi'] > 1.0].copy()

total_maker_cost = df_maker['maker_total_cost'].sum()
total_maker_ev   = df_maker['maker_total_cost'].sum() + df_maker['maker_total_edge'].sum()
total_fee_saving = df_maker['fee_saving'].sum()
print(f"Maker orders  |  Total cost ${total_maker_cost:.0f}  |  "
      f"Expected edge ${df_maker['maker_total_edge'].sum():.2f}  |  "
      f"ROI {(total_maker_ev / total_maker_cost - 1)*100:.1f}%  |  "
      f"Fee saving vs taker ${total_fee_saving:.2f}\n")

df_maker[['Market Ticker','Market Title', 'Side', 'Ask Price ($)', 'maker_ask', 'maker_order_price', 'EV',
          'maker_roi', 'additional_shares', 'maker_total_cost', 'maker_total_edge', 'fee_saving']]\
    .sort_values('Market Ticker', ascending=False)\
    .style.format({'Ask Price ($)': '{:.2f}', 'maker_ask': '{:.2f}', 'maker_order_price': '{:.3f}',
                   'EV': '{:.3f}', 'maker_roi': '{:.3f}',
                   'maker_total_cost': '${:.2f}', 'maker_total_edge': '${:.2f}',
                   'fee_saving': '${:.2f}'})

Maker orders  |  Total cost $257  |  Expected edge $43.29  |  ROI 16.8%  |  Fee saving vs taker $9.42



,Market Ticker,Market Title,Side,Ask Price ($),maker_ask,maker_order_price,EV,maker_roi,additional_shares,maker_total_cost,maker_total_edge,fee_saving
845,KXTITLEDTUESTOP-26JUL21T8-VART,"Will Vladislav Artemiev finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,0.88,0.87,0.872,0.976,1.119,65,$56.68,$6.74,$1.00
1075,KXTITLEDTUESTOP-26JUL21T8-JMAR,"Will Jose Martinez finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",NO,0.90,0.89,0.892,1.000,1.121,174,$155.16,$18.84,$2.54
1205,KXTITLEDTUESTOP-26JUL21T8-DAND,"Will Dmitry Andreikin finish top 8 in the Titled Tuesday chess competition on Jul 21, 2026?",YES,0.18,0.17,0.172,0.238,1.379,101,$17.42,$6.60,$1.80
553,KXTITLEDTUESTOP-26JUL21T3-PMAG,"Will Parham Maghsoodloo finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",YES,0.10,0.09,0.091,0.116,1.265,57,$5.21,$1.38,$0.85
724,KXTITLEDTUESTOP-26JUL21T3-DLAZ,"Will Denis Lazavik finish top 3 in the Titled Tuesday chess competition on Jul 21, 2026?",YES,0.14,0.13,0.132,0.190,1.436,150,$19.80,$8.64,$2.47
260,KXTITLEDTUESDAY-26JUL21-JDUD,"Will Jan-Krzysztof Duda win the Titled Tuesday weekly chess competition, originally scheduled for Jul 21, 2026 at 11:00am EDT?",YES,0.06,0.05,0.051,0.070,1.368,58,$2.95,$1.08,$0.76
